# SynBioCrow_M2_4_12_BIOPKS_AUTOMATED_NATIVE_RELEASE

**Goal:** Automatically connect native BioPKS `designPKS` candidates to native thioesterase release and measure exact target product identity.

**Context:** Native PKS designs and a separately invoked native TE release previously worked, but upstream `designPKS` did not select active TE.

**Constraints:** Do not claim upstream automatic TE selection, balance, or certified routes. BioPKS and RetroTide are one implementation family, not two independent engines. No previous uploads.

**Done when:** A fresh Colab run exports candidate designs, automatic adapter-invoked native TE transformations, exact target matches, and a notebook-matched results ZIP.

**Seventh engine:** Unresolved; do not fabricate or double-count RetroTide.


In [ ]:
from pathlib import Path
import json,subprocess,zipfile,shutil
ROOT=Path('/content/synbiocrow_m2412');ROOT.mkdir(parents=True,exist_ok=True)


In [ ]:
# Self-healing isolated BioPKS environment. No manual uploads or prior cells required.
import subprocess, shutil, urllib.request, time, os, json
from pathlib import Path
ROOT_ENV=Path('/content/biosynthesis_ensemble/engines/biopks_retrotide')
ENV=ROOT_ENV/'env'; PY=ENV/'bin/python'; REPO=ROOT_ENV/'source'; LOG=ROOT/'install.log'
ROOT_ENV.mkdir(parents=True,exist_ok=True)
PIN='372cbfe9caf452fdf83976e6d0f32511dd28fccb'
def cmd(args,timeout=2400):
 print('RUN', ' '.join(map(str,args)),flush=True)
 with LOG.open('a') as f:
  p=subprocess.run(list(map(str,args)),stdout=f,stderr=subprocess.STDOUT,text=True,timeout=timeout)
 if p.returncode:
  raise RuntimeError(f'Command failed (exit {p.returncode}); see {LOG}')
 return p

def te_ok():
 if not PY.is_file():return False
 p=subprocess.run([str(PY),'-c','from biopks_pipeline.retrotide.bcs import TE; from rdkit import Chem; print("TE_OK")'],capture_output=True,text=True,timeout=120)
 print('Environment probe:',p.returncode,p.stdout.strip(),p.stderr[-500:])
 return p.returncode==0
if not te_ok():
 print('Isolated BioPKS environment missing/incomplete; provisioning automatically.')
 MM=Path('/content/micromamba')
 if not MM.is_file():
  archive=ROOT/'micromamba.tar.bz2'
  urllib.request.urlretrieve('https://micro.mamba.pm/api/micromamba/linux-64/latest',archive)
  cmd(['tar','-xjf',archive,'-C',str(ROOT),'bin/micromamba'])
  shutil.copy(ROOT/'bin/micromamba',MM)
 if not PY.is_file():cmd([str(MM),'create','-y','-p',str(ENV),'-c','conda-forge','python=3.10','pip','rdkit'],timeout=3600)
 if not (REPO/'.git').exists():cmd(['git','clone','https://github.com/JBEI/BioPKS-Pipeline.git',str(REPO)],timeout=900)
 cmd(['git','-C',str(REPO),'checkout','--detach',PIN],timeout=180)
 actual=subprocess.check_output(['git','-C',str(REPO),'rev-parse','HEAD'],text=True).strip()
 assert actual==PIN,('Wrong upstream commit',actual)
 original=(REPO/'requirements.txt').read_text()
 filtered=[line for line in original.splitlines() if not line.strip().lower().replace(' ','').startswith('umap==0.1.1')]
 if len(filtered)==len(original.splitlines()):raise RuntimeError('Expected umap==0.1.1 pin absent; review upstream dependency contract')
 compat=ROOT/'requirements_colab_compat.txt';compat.write_text('\n'.join(filtered)+'\n')
 cmd([str(PY),'-m','pip','install','--upgrade','pip','wheel'],timeout=900)
 cmd([str(PY),'-m','pip','install','setuptools==80.9.0'],timeout=900)
 cmd([str(PY),'-m','pip','install','-r',str(compat)],timeout=3600)
 cmd([str(PY),'-m','pip','install','--no-deps','--no-build-isolation','-e',str(REPO)],timeout=1200)
 cmd([str(PY),'-m','pip','install','setuptools==80.9.0'],timeout=900)
 if not te_ok():raise RuntimeError('Native BioPKS TE still unavailable after automatic installation; see install.log')
print('Native TE ready:',PY)


In [ ]:
RUNNER="#!/usr/bin/env python3\nimport json,traceback,sys\nfrom pathlib import Path\nfrom rdkit import Chem\nfrom biopks_pipeline.retrotide.retrotide import designPKS\nfrom biopks_pipeline.retrotide.bcs import TE\n\ndef canon(s):\n m=Chem.MolFromSmiles(s)\n return Chem.MolToSmiles(m,isomericSmiles=True) if m is not None else None\n\ndef main(out):\n out=Path(out);out.mkdir(parents=True,exist_ok=True)\n targets=[('hexanoic_acid','CCCCCC(=O)O'),('3_hydroxypropionic_acid','O=C(O)CCO'),('butanal','CCCC=O')]\n te=TE(active=True,cyclic=False,ring=0)\n all_records=[]; reports=[]\n for name,target in targets:\n  report={'target':name,'smiles':target,'design_count':0,'native_te_calls':0,'exact_product_matches':0,'errors':[]}\n  try:\n   rounds=designPKS(Chem.MolFromSmiles(target),maxDesignsPerRound=5,similarity='atompairs')\n   for ridx,group in enumerate(rounds):\n    for didx,(cluster,score,bound) in enumerate(group):\n     report['design_count']+=1\n     rec={'target':name,'target_smiles':target,'round':ridx+1,'design_index':didx,'score':float(score),'native_bound_smiles':Chem.MolToSmiles(bound,isomericSmiles=True),'native_cluster_repr':repr(cluster)[:2500],'release_selection':'SYBNIOCROW_ADAPTER_APPLIED_NATIVE_TE_TO_NATIVE_DESIGN','upstream_designPKS_selected_active_TE':False,'balanced_reaction_certified':False,'route_certified':False,'native_te_called':False,'exact_target_match':False}\n     try:\n      released=te.operation(Chem.Mol(bound))\n      report['native_te_calls']+=1\n      rec['native_te_called']=True\n      rec['native_release_product_smiles']=Chem.MolToSmiles(released,isomericSmiles=True)\n      rec['exact_target_match']=canon(rec['native_release_product_smiles'])==canon(target)\n      rec['native_te_cofactors']={'reactants':[x.id for x in te.reactants()],'products':[x.id for x in te.products()]}\n      if rec['exact_target_match']:report['exact_product_matches']+=1\n     except Exception as exc:\n      rec['release_error']=repr(exc);report['errors'].append(repr(exc))\n     all_records.append(rec)\n  except Exception as exc:report['errors'].append(traceback.format_exc()[-6000:])\n  reports.append(report)\n (out/'native_design_release_records.json').write_text(json.dumps(all_records,indent=2))\n (out/'target_reports.json').write_text(json.dumps(reports,indent=2))\n summary={'milestone':'M2.4.12','native_designs':sum(x['design_count'] for x in reports),'native_te_calls':sum(x['native_te_calls'] for x in reports),'exact_target_matches':sum(x['exact_product_matches'] for x in reports),'gate':'ADAPTER_ORCHESTRATED_NATIVE_REACTION_PASS' if any(x['exact_product_matches'] for x in reports) else 'NOT_VERIFIED','upstream_automatic_TE_selection_verified':False,'note':'Native designPKS output passed automatically to native TE.operation by SynBioCrow adapter. This is NOT upstream automatic TE design selection, and is not a balanced reaction certification.'}\n (out/'summary.json').write_text(json.dumps(summary,indent=2));print(json.dumps(summary,indent=2))\nif __name__=='__main__':main(sys.argv[1])\n"
(ROOT/'runner.py').write_text(RUNNER)
OUT=ROOT/'outputs';OUT.mkdir(exist_ok=True)
p=subprocess.run([str(PY),str(ROOT/'runner.py'),str(OUT)],capture_output=True,text=True,timeout=1200)
(OUT/'stdout.log').write_text(p.stdout);(OUT/'stderr.log').write_text(p.stderr)
if LOG.exists():shutil.copy(LOG,OUT/'install.log')
print('Runner exit:',p.returncode,'\n',p.stdout[-3500:],'\n',p.stderr[-1500:])


In [ ]:
ARCHIVE=Path('/content/SynBioCrow_M2_4_12_BIOPKS_AUTOMATED_NATIVE_RELEASE_ALL_OUTPUTS.zip')
with zipfile.ZipFile(ARCHIVE,'w',zipfile.ZIP_DEFLATED) as z:
 for f in OUT.rglob('*'):
  if f.is_file():z.write(f,f.relative_to(OUT))
print('Saved',ARCHIVE)
try:
 from google.colab import files
 files.download(str(ARCHIVE))
except Exception as e:print('Automatic download unavailable:',e)


In [ ]:
# Rescue download
from google.colab import files
files.download(str(ARCHIVE))
